In [1]:
from data import *
from data_utils import add_corresponding_terms, add_edges

Correspondances trouvées : 550


/home/onyxia/work/OT_simulation_maladies/Embeddings/data.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  profils_omim = profils_omim.reset_index()


In [13]:
import networkx as nx
import numpy as np

l = [len(nx.descendants(G_hpo_work, n)) for n in G_hpo_work.nodes()]
np.unique(l)

list(G_hpo_work.predecessors('HP:0000001'))

['HP:0000005',
 'HP:0000118',
 'HP:0012823',
 'HP:0020228',
 'HP:0032223',
 'HP:0032443',
 'HP:0040279']

In [ ]:
print('Union diseases')
union_diseases = add_corresponding_terms(work_omim, work_orpha, df_orpha_omim)
print('Add edges')
G_hpo_omim = add_edges(union_diseases, G_hpo_work, depths)
print("End of preprocessing")
G_hpo_omim.add_edge('HP:6001347', 'HP:0001832')

label_to_id = {node: str(i) for i, node in enumerate(G_hpo_omim.nodes())}
id_to_label = {v: k for k, v in label_to_id.items()}
G_clean = nx.relabel_nodes(G_hpo_omim, label_to_id)  # Nouveaux labels pour éviter les pollutions

objects = list(G_hpo_omim.nodes())
edges = np.array([(label_to_id[u], label_to_id[v]) for u, v in G_hpo_omim.edges()],dtype=int)
#nx.write_edgelist(G_clean, "graph.edg", delimiter="\t", data=False)
#nx.write_edgelist(G_hpo_omim, "graph.edg", delimiter="\t", data=False)

Union diseases


Fusion HPO: 100%|██████████| 5312/5312 [00:00<00:00, 6131.79it/s]


Add edges


100%|██████████| 6562/6562 [00:11<00:00, 560.81it/s]


End of preprocessing


In [3]:
nodes_to_keep = [c for c in union_diseases.columns if c.startswith('HP')]
G_sub = G_hpo_omim.subgraph(nodes_to_keep).copy()

print(f"Noeuds : {G_hpo_omim.number_of_nodes()} → {G_sub.number_of_nodes()}")
print(f"Arêtes : {G_hpo_omim.number_of_edges()} → {G_sub.number_of_edges()}")

Noeuds : 19391 → 11613
Arêtes : 2186608 → 2174438


In [3]:
import numpy as np
import pandas as pd

nodes = []
vectors = []

with open("graph.emb", "r") as f:
    next(f)
    i=0
    for line in f:
        i+=1
        parts = line.strip().split()
        try:
            vec = np.array(parts[1:], dtype=np.float32)
            nodes.append(parts[0])
            vectors.append(vec)
        except ValueError:
            print(f"Ligne {i} ignorée : {line[:]}")
df = pd.DataFrame(vectors, index=nodes)
df.index = [int(label_to_id[i]) for i in df.index]
df_sorted = df.sort_index()
max_id = df_sorted.index.max()
df_sorted = df_sorted.reindex(range(max_id + 1))
ignored_label = [c for c in id_to_label.keys() if not np.isin(int(c), df_sorted.index)]

Ligne 17214 ignorée : Comment: Affected individuals may present with foot pain on weight-bearing, swelling, and tenderness. Radiological features include widening of the metatarsophalangeal joint space and flattening of the affected metatarsal head, at later stages metatarsal head sclerosis, cortical thickening, and intra-articular loose bodies. Osteochondrosis is a condition resulting from an epiphysis injury that alters enchondral ossification and produces irregularity at the joint surface. Freiberg disease is characterized as osteochondrosis of the metatarsal heads (primarily the second)."", xrefs=('PMID:30685014', 'PMID:30725993')"")" 0.0821505 0.012895118 -0.067146435 -0.11278694 0.105025284 0.03998919 0.020470139 0.032430608 0.10429149 -0.099818945 0.06482978 -0.02671577 0.112023905 0.03470147 0.04848226 0.10817434 -0.09108107 0.02597665 -0.018610151 -0.07946276 -0.019600997 -0.004767047 -0.007365023 -0.086254686 -0.022665929 -0.06743415 0.052404623 -0.16472206 -0.019311633 -0.03

In [40]:
from collections import defaultdict
from tqdm import tqdm
from sklearn.metrics import average_precision_score

def evaluate(weights, objects, edges, node2id):

    pos_neighbors = defaultdict(set)
    for u, v in edges:
        pos_neighbors[int(u)].add(int(v))
    w = weights.values
    n = w.shape[0]
    ap_scores = []
    ranks_all = []
    labels = np.zeros(n)

    for obj in tqdm(objects):
        u = int(node2id[obj])  # Identifiant du terme 
        neighbors = pos_neighbors.get(u, set())
        if not neighbors:
            continue
        if np.any(np.isnan(w[u])):
            continue
        u_emb = np.tile(w[u], (n, 1))
        dists = np.linalg.norm(u_emb-w, axis=1)
        dists[u] = float('inf')
        dists[np.isnan(dists)] = float('inf')

        max_finite = dists[np.isfinite(dists)].max()
        dists[~np.isfinite(dists)] = max_finite + 1.0

        sorted_ind = np.argsort(dists)
        ranks = np.where(np.isin(sorted_ind, list(neighbors)))[0] + 1

        n_neighbors = len(neighbors)
        corrected_ranks = ranks - np.arange(n_neighbors)
        ranks_all.extend(corrected_ranks.tolist())

        labels.fill(0)
        labels[list(neighbors)] = 1
        ap_scores.append(average_precision_score(labels, -dists))

    map_score  = float(np.mean(ap_scores))
    mean_rank  = float(np.mean(ranks_all))
    return map_score, mean_rank
  

In [41]:
results = evaluate(df_sorted, objects, edges, label_to_id)
results

100%|██████████| 19391/19391 [08:01<00:00, 40.31it/s]


(0.028388536478705838, 4453.802849255879)

In [5]:
from joblib import Parallel, delayed
from tqdm import tqdm
from OT_utils import compute_transport
from information_content import deprecated

def compute_costs_matrix_wasserstein2(df_omim, df_orpha, node2id_w, df_weights, deprecated):
    n=len(df_omim)
    m=len(df_orpha)
    hpo_cols = [c for c in df_omim.columns if c.startswith('HP:')]
    W = df_weights.values
    print(type(W))

    print("Precompute...")
    def precompute(df):
        '''
        Renvoie pour chaque maladie (ligne) du dataframe df la liste des termes HPO actifs et 
        le vecteur de poids uniformes associés.
        '''
        X = df[hpo_cols].to_numpy(dtype=bool)
        resolved_cols = np.array(
            [deprecated.get(col, col) if deprecated.get(col, col) in node2id_w else None for col in hpo_cols], 
            dtype=object)
        valid_mask = resolved_cols != None
        X_valid = X[:, valid_mask]
        resolved_valid = resolved_cols[valid_mask]
        terms = [list(resolved_valid[row_mask]) for row_mask in X_valid]
        weights = [np.ones(len(t)) / len(t) if t else np.array([]) for t in terms]
        return terms, weights

    terms_i, weights_i = precompute(df_omim)  # Termes actifs, poids pour les maladies sources
    terms_j, weights_j = precompute(df_orpha)  # Termes actifs, poids pour les maladies destinations
    print("Finished !")

    all_terms = list({h for ts in terms_i + terms_j for h in ts})  # Tous les termes actifs
    term2idx = {h: k for k, h in enumerate(all_terms)}
    missing = [h for h in all_terms if h not in node2id_w]
    print(f"Termes absents de node2id_w : {missing[:10]}")
    E = W[[int(node2id_w[h]) for h in all_terms]]  # Embeddings des termes actifs
    E = E.astype(np.float32)

    idx_i = [[term2idx[h] for h in ts] for ts in terms_i]  # Index des termes actifs par maladies sources
    idx_j = [[term2idx[h] for h in ts] for ts in terms_j]  # Index des termes actifs par maladies destinations

    C = np.zeros((n,m))

    print("Precomputing full HPO distance matrix...")
    #D_full = np.sum((E[:, None, :] - E[None, :, :]) ** 2, axis=-1)  # (K, K)
    norms = np.sum(E**2, axis=1)  # (K,)
    D_full = norms[:, None] + norms[None, :] - 2 * (E @ E.T)  # (K, K)
    D_full = np.maximum(D_full, 0)
    print(f"HPO distance matrix: {D_full.shape}")

    def compute_row(i):
        if not idx_i[i]:
            return i, np.zeros(len(terms_j))
        # Ei = E[idx_i[i]]
        row = np.zeros(len(terms_j))
        valid_js = [j for j in range(len(terms_j)) if idx_j[j]]
        for j in valid_js:
            # Ej = E[idx_j[j]]
            # M = cost_hpos(Ei, Ej) 
            M = D_full[np.ix_(idx_i[i], idx_j[j])]
            _, row[j] = compute_transport(M, weights_i[i], weights_j[j])
        return i, row
    
    results = Parallel(n_jobs=-1)(
        delayed(compute_row)(i) for i in tqdm(range(len(df_omim)), desc="OMIM")
    )
    C = np.zeros((len(df_omim), len(df_orpha)))
    for i, row in results:
        C[i] = row
    return C

C = compute_costs_matrix_wasserstein2(work_omim, work_orpha, label_to_id, df, deprecated)

<class 'numpy.ndarray'>
Precompute...
Finished !
Termes absents de node2id_w : []
Precomputing full HPO distance matrix...
HPO distance matrix: (10649, 10649)


OMIM: 100%|██████████| 6562/6562 [24:40<00:00,  4.43it/s]


In [7]:
from data_utils import f_ground_truth
w_omim = work_omim.reset_index(drop=True)
w_orpha = work_orpha.reset_index(drop=True)
_, valid_omim, valid_orpha = f_ground_truth(w_omim, w_orpha, df_orpha_omim)

w_omim = w_omim[w_omim['database_id'].isin(valid_omim)].reset_index(drop=True)
w_orpha = w_orpha[w_orpha['database_id'].isin(valid_orpha)].reset_index(drop=True)
hpo_cols0 = [c for c in w_omim.columns if c.startswith("HP:")]
gt_set, valid_omim0, valid_orpha0 = f_ground_truth(w_omim, w_orpha, df_orpha_omim)

In [9]:
from OT_utils import compute_transport, compute_transport_sinkhorn, evaluate_transport
print("======== Sans régularisation ========")
ot_plan, ot_cost = compute_transport(C, None, None)
ranks, pairs = evaluate_transport(ot_plan, gt_set, C)

print("======== Avec régularisation ========")
epsilon = 0.1*np.mean(C)
ot_plan_reg, ot_cots_reg = compute_transport_sinkhorn(C, None, None, epsilon, 10000, 1e-4, False)
ranks_reg, pairs_reg = evaluate_transport(ot_plan_reg, gt_set, C)

======== Sans régularisation ========
Paires évaluées : 5312
Top-1 accuracy : 0.000 (1/5312)
Top-3 accuracy : 0.001 (5/5312)
Top-5 accuracy : 0.001 (6/5312)
 Rang moyen: 1644.73
======== Avec régularisation ========
Paires évaluées : 5312
Top-1 accuracy : 0.000 (1/5312)
Top-3 accuracy : 0.001 (4/5312)
Top-5 accuracy : 0.001 (6/5312)
 Rang moyen: 1584.41
